In [ ]:
import pandas as pd
from llama_cpp import Llama
from eval.translation_metrics import calc_bleurt_score, calc_comet_score, calc_dbleu_score, calc_ter_score
import os
os.chdir("..")

In [ ]:


llm = Llama.from_pretrained(
    repo_id="Triangle104/EuroLLM-9B-Q8_0-GGUF",
    filename="eurollm-9b-q8_0.gguf",
)


In [ ]:
def translate(input_text):
    output = llm(
        f"""Translate the following German text to English. Only return the translated text, nothing else.\n"
        "German: {input_text}r\n"
        "English:""",
        max_tokens=50,  # Limit the response length
        stop=["\n"],  # Stops generation at the first newline to avoid extra text
        temperature=0  # Makes the output deterministic
    )
    return output['choices'][0]['text']


In [ ]:


df = pd.read_csv('data/new_sample.csv')

In [ ]:
df['en_transalted'] = df['de'].apply(translate)

## Apply all the transaltion metrics.

In [ ]:
def calc_metrics(org_col: str , machine_translated_col: str, gd_translated_col: str, df: pd.DataFrame):
    '''
    This function will calculate the metrics for the given columns
    :param org_col: the column the text in original language
    :param machine_translated_col: the column with the machine translated text
    :param gd_translated_col: the column with the human translated text
    :param df: the dataframe with the data.
    :return:
    '''
    df['dbleu'] = df.apply(lambda x: calc_dbleu_score(x[gd_translated_col].lower(), x[machine_translated_col].lower()), axis=1)
    df['bleurt'] = df.apply(lambda x: calc_bleurt_score(x[gd_translated_col].lower(), x[machine_translated_col].lower()), axis=1)
    df['comet'] = df.apply(lambda x: calc_comet_score(x[org_col].lower(), x[gd_translated_col].lower(), x[machine_translated_col].lower()), axis=1)

    df[['ter_num_edits','ter_score']]  = df.apply(lambda x: pd.Series(calc_ter_score(x[gd_translated_col].lower(), x[machine_translated_col].lower())), axis=1)
    df['dbleu'] = df['dbleu'].astype(float)
    return df


In [ ]:
df = calc_metrics('de', 'en_transalted', 'en', df)

In [ ]:
df.head()

In [ ]:
df[[ 'dbleu', 'comet', 'bleurt', 'ter_num_edits']].describe()

In [ ]:
def summarize_df(df):

    output = llm(
        f"""You are an AI assistant. Your task is to summarize the given pandas DataFrame into the main points. Please provide a concise summary for this dataframe :{df}""",
        max_tokens=512,  # Limit the response length
        stop=["\n"],  # Stops generation at the first newline to avoid extra text
        temperature=0  # Makes the output deterministic
    )
    return output['choices'][0]['text']


In [25]:
df[[ 'dbleu', 'comet', 'bleurt', 'ter_num_edits']].describe()

,dbleu,comet,bleurt,ter_num_edits
count,50.000000,50.000000,50.000000,18.000000
mean,14.038990,0.556420,-0.953566,10.111111
std,24.610616,0.216739,0.927806,9.361303
min,0.000000,0.335561,-2.312927,0.000000
25%,0.000000,0.397738,-1.641890,3.250000
50%,0.000000,0.424278,-1.246623,7.000000
75%,18.180648,0.812985,0.030089,17.500000
max,100.000000,0.973265,0.949728,31.000000


- dbleu: Shows a wide range of scores with high variability, indicating significant differences in performance.
- comet: Has a moderate average score with low to moderate variability, suggesting more consistent performance.
- bleurt: Displays negative average scores with moderate to high variability, indicating a mix of positive and negative performance.
- ter_num_edits: Shows moderate average scores with high variability, reflecting a wide range of edit distances.

In [24]:
summarize_df(df[[ 'dbleu', 'comet', 'bleurt', 'ter_num_edits']].describe())

Llama.generate: 375 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    6772.80 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =     128.73 ms /     1 runs   (  128.73 ms per token,     7.77 tokens per second)
llama_perf_context_print:       total time =     129.98 ms /     2 tokens


''